In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
import os
import shutil
from sklearn.metrics import accuracy_score

%matplotlib inline
seed = 0
np.random.seed(seed)

tf.random.set_seed(seed)

os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

2026-04-30 11:45:43.157473: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_test = X_test.astype("float32") / 255
y_test = keras.utils.to_categorical(y_test, 10)

In [3]:
from qkeras.utils import load_qmodel

# This replaces the standard keras load_model
model = load_qmodel('Model_Qkeras_6bw_pruned.h5')

score = model.evaluate(X_test, y_test)

ImportError: cannot import name 'QAdaptiveActivation' from 'qkeras.qlayers' (/home/ncgadmin/miniconda3/envs/DAT255_SynthVitis/lib/python3.12/site-packages/qkeras/qlayers/__init__.py)

In [ ]:
import keras

inputs = keras.Input(shape=(32, 32, 3), name="hw_input")


x = model.get_layer('qconv0')(inputs)
x = model.get_layer('batch_normalization')(x) 
x = model.get_layer('relu0')(x)
x = model.get_layer('pool0')(x)

x = model.get_layer('qconv1')(x)
x = model.get_layer('batch_normalization_1')(x) 
x = model.get_layer('relu1')(x)
x = model.get_layer('pool1')(x)

x = model.get_layer('qconv2')(x)
x = model.get_layer('batch_normalization_2')(x)
x = model.get_layer('relu2')(x)
x = model.get_layer('pool2')(x)

x = keras.layers.Flatten()(x)

x = model.get_layer('qdense0')(x)
x = model.get_layer('relu3')(x)

# We can keep Dropout here; hls4ml will automatically ignore it during conversion
x = model.get_layer('dropout')(x) 

x = model.get_layer('qdense1')(x)
outputs = model.get_layer('softmax')(x)

# 3. Create the stripped model
model_stripped = keras.Model(inputs=inputs, outputs=outputs)

# 4. Verify the weights are transferred correctly
# Usually, model.get_layer() handles the weight references, 
# but a quick summary and test is always good practice.
model_stripped.summary()

ValueError: No such layer: qconv0. Existing layers are: ['input_5', 'qconv1', 'relu1', 'pool1', 'qconv2', 'relu2', 'pool2', 'flatten_4', 'dropout_4', 'qdense1', 'softmax'].

In [4]:
import hls4ml

output_dir ="/home/ncgadmin/DAT255/DAT255-project/MNIST/Qkeras/_HLS4ML"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    
hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')

#hls_config['Model']['Strategy'] = 'Distributed Arithmetic'
#proj_name = f"{str(model_to_test)}_{str(model_revision)}_hls4ml_prj_{str(hls4ml_revision)}"

hls_model = hls4ml.converters.convert_from_keras_model(
    model,    
    backend='Vitis',
    hls_config=hls_config,
    io_type='io_stream',
    #proj_name = proj_name,
    output_dir=output_dir, 
    board       = 'kv260',
    part='xck26-sfvc784-2LV-c',
    clock_period='5',
)
hls_model.compile()

/home/ncgadmin/miniconda3/envs/hls4ml-tutorial_HGQ+qkeras/lib/python3.10/site-packages/keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(


In [5]:
hls_model.build(csim=False, synth=True, cosim=False)


****** vitis-run v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Tue Apr 28 20:15:22 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
Sourcing Tcl script '/home/ncgadmin/DAT255/DAT255-project/MNIST/Qkeras/hls4ml_prj_stream_latency/build_prj.tcl'
INFO: [HLS 200-1510] Running: open_project myproject_prj 
Resolution: For help on HLS 200-2182 see docs.amd.com/access/sources/dita/topic?Doc_Version=2025.2%20English&url=ug1448-hls-guidance&resourceid=200-2182.html
INFO: [HLS 200-10] Creating and opening solution '/home/ncgadmin/DAT255/DAT255-project/MNIST/Qkeras/hls4ml_prj_stream_latency/myproject_prj'.
INFO: [HLS 200-1510] Running: set_top myproject 
INFO: [HLS 200-1510] Running: add_files firmware/myproject.cpp -cflags -std=c++0x 
INFO: [HLS 200-10] Adding design file 'firmware/myproject.cpp' to the project
INFO: [HLS 20

{'CSynthesisReport': {'TargetClockPeriod': '5.00',
  'EstimatedClockPeriod': '3.646',
  'BestLatency': '2367',
  'WorstLatency': '2367',
  'IntervalMin': '2352',
  'IntervalMax': '2352',
  'BRAM_18K': '134',
  'DSP': '46',
  'FF': '34445',
  'LUT': '143665',
  'URAM': '0',
  'AvailableBRAM_18K': '288',
  'AvailableDSP': '1248',
  'AvailableFF': '234240',
  'AvailableLUT': '117120',
  'AvailableURAM': '64'}}